In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR=Path("../../data/processed")
OUTPUT_DIR=Path("../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
invoices =pd.read_csv(DATA_DIR/ "invoices.csv")
contract_rates =pd.read_csv(DATA_DIR /"contract_rates.csv")
shipment_evidence=pd.read_csv(DATA_DIR/ "shipment_evidence.csv")
load_evidence =pd.read_csv(DATA_DIR/"load_evidence.csv")
trip_evidence= pd.read_csv(DATA_DIR / "trip_evidence.csv")

In [ ]:
print("INVOICES")
print(invoices.columns.tolist())
print("\nCONTRACT RATES")
print(contract_rates.columns.tolist())
print("\nSHIPMENT EVIDENCE")
print(shipment_evidence.columns.tolist())
print("\nLOAD EVIDENCE")
print(load_evidence.columns.tolist())
print("\nTRIP EVIDNCE")
print(trip_evidence.columns.tolist())


In [ ]:
print("Duplicate invoice IDs:",
      invoices["invoice_id"].duplicated().sum())
print("Duplicate shipment Ids:",
      shipment_evidence["shipment_id"].duplicated().sum())
print("Duplicate load IDs:",
      load_evidence["load_id"].duplicated().sum())
print("Duplicate trip ID:",
      trip_evidence["trip_id"].duplicated().sum())
print("Duplicate contract IDs:",
      contract_rates["contract_id"].duplicated().sum())

In [ ]:
contract_cols=[
    "contract_id",
    "route_id",
   "contract_rate_per_mile",
    "contract_fuel_surcharge_rate",
    "typical_distance_miles",
    "typical_transit_days",
    "valid_from",
    "valid_to"
]
contract_data=contract_rates[contract_cols].copy()

In [ ]:
shipment_cols =[
    "shipment_id",
    "load_id",
    "trip_id",
    "customer_id",
    "route_id",
    "load_date",
    "load_type",
    "weight_lbs",
    "pieces",
    "typical_distance_miles",
    "actual_distance_miles",
    "base_rate_per_mile",
    "fuel_surcharge_rate",
    "detention_minutes",
    "fuel_gallons_used",
    "actual_duration_hours",
    "average_mpg",
    "idle_time_hours"
]
shipment_data =shipment_evidence[shipment_cols].copy()

In [ ]:
load_cols =[
    "load_id",
    "customer_id",
    "route_id",
    "load_date",
    "load_type",
    "weight_lbs",
    "pieces",
    "revenue",
    "fuel_surcharge",
    "accessorial_charges",
    "load_status",
    "booking_type"
]
load_data= load_evidence[load_cols].copy()


In [ ]:
trip_cols= [
    "trip_id",
    "load_id",
    "actual_distance_miles",
    "actual_duration_hours",
    "fuel_gallons_used",
    "average_mpg",
    "idle_time_hours",
    "trip_status"
]
trip_data=trip_evidence[trip_cols].copy()


In [ ]:
invoice_cols =[
    "invoice_id",
    "shipment_id",
    "load_id",
    "customer_id",
    "route_id",
    "contract_id",
    "invoice_date",
    "actual_distance_miles",
    "base_rate_per_mile",
    "calculated_base_charge",
    "fuel_surcharge_rate",
    "calculated_fuel_surcharge",
    "detention_minutes",
    "detention_free_minutes",
    "detention_rate_per_hour",
    "calculated_detention_charge",
    "calculated_accessorial_charges",
    "calculated_subtotal",
    "minimum_charge",
    "expected_total",
    "billed_amount"
]
invoice_data=invoices[invoice_cols].copy()


In [ ]:
training_data =invoice_data.merge(
    contract_data,
    on=["contract_id", "route_id"],
    how="left",
    suffixes=("","_contract")
)

In [ ]:
training_data =training_data.merge(
    shipment_data,
    on=["shipment_id", "load_id", "customer_id", "route_id"],
    how="left",
    suffixes=("","_shipment")
)

In [ ]:
training_data = training_data.merge(
    load_data,
    on=["load_id","customer_id", "route_id"],
    how="left",
    suffixes=("", "_load")
)

In [ ]:
training_data = training_data.merge(
    trip_data,
    on=["trip_id", "load_id"],
    how="left",
    suffixes=("","_trip")
)

In [ ]:
print("Final shape:", training_data.shape)

print("\nColumns:")
for column in training_data.columns:
    print(column)

print("No of invoices:", training_data["invoice_id"].nunique())
print("No. of rows:", len(training_data))

In [ ]:
missing=training_data.isnull().sum()
missing=missing[missing > 0].sort_values(ascending=False)
missing

In [ ]:
duplicate_invoices=training_data[
    training_data["invoice_id"].duplicated(keep=False)
]

In [ ]:
output_path=OUTPUT_DIR/"prepared_training_data.csv"

training_data.to_csv(output_path, index=False)

print(f"Prepared dataset saved to: {output_path}")